### Импортирование библиотек

In [ ]:
import pandas as pd
import numpy as np

from collections import defaultdict
from copy import deepcopy
from sklearn.utils import shuffle
from scipy.sparse import dok_matrix

### Чтение данных и их подготовка

In [ ]:
df = pd.read_csv('misis_df_v2.csv', index_col='Timestamp')
df = df.reset_index(drop=True)
df.index.name = 'Id'
df.head(3)

,Введите Ваш возраст,Выберите Ваш пол,Выберите Ваш курс обучения,Выберите Вашу сферу обучения,"Выберите 5 форматов мероприятий, которые Вы бы хотели посещать в университете МИСиС."
Id,,,,,
0,19,Мужской,"2 курс (бакалавриат, специалитет)",IT,Хакатоны;Предпринимательство;Стажировки;Робото...
1,19,Мужской,"2 курс (бакалавриат, специалитет)",IT,Backend-разработка;Хакатоны;DevOps;Стажировки;...
2,19,Мужской,"2 курс (бакалавриат, специалитет)",IT,Хакатоны;Машинное обучение;Глубокое обучение;А...


In [ ]:
df = pd.DataFrame(df['Выберите 5 форматов мероприятий, которые Вы бы хотели посещать в университете МИСиС.'])
df['user_id'] = df.index + 1
df.rename(columns={'Выберите 5 форматов мероприятий, которые Вы бы хотели посещать в университете МИСиС.': 'items'}, inplace=True)

In [ ]:
split_items = df['items'].str.split(';', expand=True)
split_items.columns = [f'item_{i}' for i in range(len(split_items.columns))]
df = pd.concat([df, split_items], axis=1)
df.drop(columns=['items'], inplace=True)
df

,user_id,item_0,item_1,item_2,item_3,item_4
Id,,,,,,
0,1,Хакатоны,Предпринимательство,Стажировки,Робототехника,Иностранные языки
1,2,Backend-разработка,Хакатоны,DevOps,Стажировки,Спортивное программирование
2,3,Хакатоны,Машинное обучение,Глубокое обучение,Анализ данных и Big Data,Стажировки
3,4,Backend-разработка,Хакатоны,Машинное обучение,Стажировки,Спорт
4,5,Хакатоны,Frontend разработка,Спорт,Творчество,Туризм
...,...,...,...,...,...,...
155,156,Спорт,Творчество,Иностранные языки,Волонтерство,Туризм
156,157,Хакатоны,Машинное обучение,DevOps,Танцы,Творчество
157,158,Хакатоны,Машинное обучение,Анализ данных и Big Data,КВН,Стажировки


In [ ]:
items = set()
for i in range(5):
    items = items | set(df[f'item_{i}'].unique().tolist())
items = list(items)
items_ids = defaultdict(int)
for i in range(len(items)):
    items_ids[items[i]] = i + 1

In [ ]:
for i in range(5):
    df[f'item_{i}'] = df[f'item_{i}'].replace(items_ids)
df

/tmp/ipython-input-3640975482.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[f'item_{i}'] = df[f'item_{i}'].replace(items_ids)


,user_id,item_0,item_1,item_2,item_3,item_4
Id,,,,,,
0,1,15,28,22,13,16
1,2,6,15,12,22,27
2,3,15,24,5,30,22
3,4,6,15,24,22,33
4,5,15,8,33,25,11
...,...,...,...,...,...,...
155,156,33,25,16,14,11
156,157,15,24,12,23,25
157,158,15,24,30,34,22


In [ ]:
df_melted = df.melt(
    id_vars=['user_id'],
    value_vars=['item_0', 'item_1', 'item_2', 'item_3', 'item_4'],
    var_name='item_order',
    value_name='item_id'
)
df = df_melted[['user_id', 'item_id']].sort_values('user_id').reset_index(drop=True)
df

,user_id,item_id
0,1,13
1,1,16
2,1,22
3,1,28
4,1,15
...,...,...
795,160,11
796,160,14
797,160,16
798,160,7


 Так как некоторые категории смещены ближе к концу/началу "окна" user_id, то будет логичнее перемешать их внутри "окна" для разбиения на train и test выборки

In [ ]:
def shuffle_group(group):
    return shuffle(group, random_state=42)

In [ ]:
df = df.groupby('user_id', group_keys=False).apply(shuffle_group).reset_index(drop=True)
df

/tmp/ipython-input-979808837.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('user_id', group_keys=False).apply(shuffle_group).reset_index(drop=True)


,user_id,item_id
0,1,16
1,1,15
2,1,22
3,1,13
4,1,28
...,...,...
795,160,14
796,160,28
797,160,16
798,160,11


In [ ]:
train = df.groupby('user_id').head(4).reset_index(drop=True)
test = df.groupby('user_id').tail(1).reset_index(drop=True)

In [ ]:
test_users = list(test['user_id'])

### Функции для подсчета метрик

In [ ]:
def leave_top_k(pred, k, group_by_col='user_id', order_by_col='rating'):
    if pred.groupby(group_by_col)[group_by_col].count().max() <= k:
        return pred
    cropped_pred = deepcopy(pred)
    cropped_pred['rank'] = (cropped_pred
                           .groupby(group_by_col)[[order_by_col]]
                           .rank(method='first', ascending=False))
    cropped_pred = cropped_pred[cropped_pred['rank'] <= k].drop(columns=['rank'])
    return cropped_pred

In [ ]:
def user_hitrate(row):
  for item in row['pred_list']:
    if item in row['gt_list']:
      return 1
  return 0

In [ ]:
def coverage(pred, k, all_items=train['item_id']):
  pred_to_consider = set(leave_top_k(pred, k)['item_id'].values)
  all_items = set(all_items.values)
  return len(pred_to_consider & all_items) / len(all_items)

In [ ]:
def metric_wrap(pred, grount_truth, k, metric_by_user):
  pred_cropped = leave_top_k(pred, k)
  pred_grouped = (pred_cropped
                  .sort_values(['user_id', 'rating'], ascending=[False, False])
                  .groupby('user_id')['item_id']
                  .apply(list).rename('pred_list')
                  )
  gt_grouped = grount_truth.groupby('user_id')['item_id'].apply(list).rename('gt_list')
  to_compare = gt_grouped.to_frame().join(pred_grouped, how='left')
  to_compare['pred_list'] = to_compare['pred_list'].apply(lambda x: x if isinstance(x, list) else [])
  metric_by_user = to_compare.apply(metric_by_user, axis=1)
  return metric_by_user.mean(), metric_by_user

In [ ]:
def measure(pred, true, k, name, df=None, cov_items=train['item_id']):
  if df is None:
    df = pd.DataFrame(columns=['hit_rate@K', 'coverage@K'])
  df.loc[name, 'hit_rate@K'] = metric_wrap(pred=pred, grount_truth=true, k=k, metric_by_user=user_hitrate)[0]
  if cov_items is not None:
    df.loc[name, 'coverage@K'] = coverage(pred=pred, k=K)
  return df

### Baseline

Бейзлайн представляет из себя рекомендацию только самых популярных категорий

In [ ]:
K = 5
N = 5

In [ ]:
popular_items = train['item_id'].value_counts().head(N).index
popular_items

Index([15, 24, 22, 30, 5], dtype='int64', name='item_id')

In [ ]:
users = []
items = []
ratings = []

for user in test_users:
  users.extend([user] * N)
  items.extend(popular_items)
  ratings.extend([1] * N)

popular_preds = pd.DataFrame({'user_id': users, 'item_id': items, 'rating': ratings})
popular_preds

,user_id,item_id,rating
0,1,15,1
1,1,24,1
2,1,22,1
3,1,30,1
4,1,5,1
...,...,...,...
795,160,15,1
796,160,24,1
797,160,22,1
798,160,30,1


In [ ]:
pop_best_hr_K = 0
pop_best_cov_K = 0
max_hr = 0
max_cov = 0
name = 'MostPopularRecs'
for k in range(1, 5):
    metrics = measure(popular_preds, test, k, name)
    if metrics.loc[name, 'hit_rate@K'] >= max_hr:
        max_hr = metrics.loc[name, 'hit_rate@K']
        pop_best_hr_K = k
    if metrics.loc[name, 'coverage@K'] >= max_cov:
        max_cov = metrics.loc[name, 'coverage@K']
        pop_best_cov_K = k

print('The best hit_rate@K:')
display(measure(popular_preds, test, pop_best_hr_K, name))

print()

print('The best coverage@K:')
display(measure(popular_preds, test, pop_best_cov_K, name))

The best hit_rate@K:


,hit_rate@K,coverage@K
MostPopularRecs,0.4125,0.142857



The best coverage@K:


,hit_rate@K,coverage@K
MostPopularRecs,0.4125,0.142857


###EASE

Добавим фиктивный столбец rating, т.к. нам нужны рейтинги для работы top@k

In [ ]:
train['rating'] = np.ones(train.shape[0])
train

,user_id,item_id,rating
0,1,16,1.0
1,1,15,1.0
2,1,22,1.0
3,1,13,1.0
4,2,12,1.0
...,...,...,...
635,159,25,1.0
636,160,14,1.0
637,160,28,1.0
638,160,16,1.0


In [ ]:
def compute_weight_matrix(rating_matrix, lambd=1000):
  P = np.linalg.inv(rating_matrix.T @ rating_matrix + lambd * np.eye(rating_matrix.shape[1]))

  weight_matrix = - P / np.diag(P) + np.eye(P.shape[1])

  return weight_matrix

In [ ]:
user_num = train["user_id"].max() + 1
item_num = train["item_id"].max() + 1

rating_matrix = dok_matrix((user_num, item_num), dtype=np.float32)
for _, user, item, rating in train[['user_id', 'item_id', 'rating']].itertuples():
    rating_matrix[user, item] = float(rating)

In [ ]:
rating_matrix.shape

(161, 36)

In [ ]:
weight_matrix = compute_weight_matrix(rating_matrix)
weight_matrix

matrix([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  9.81039824e-04, ...,
          8.37900785e-04, -3.67471127e-05,  2.88664136e-03],
        [ 0.00000000e+00,  9.91738474e-04,  0.00000000e+00, ...,
          9.82509452e-04, -3.49616027e-06, -1.08315241e-05],
        ...,
        [ 0.00000000e+00,  8.24350539e-04,  9.56192960e-04, ...,
          0.00000000e+00, -1.07784332e-04,  2.72540453e-03],
        [ 0.00000000e+00, -3.68915887e-05, -3.47204181e-06, ...,
         -1.09986769e-04,  0.00000000e+00,  9.35982213e-04],
        [ 0.00000000e+00,  2.86990431e-03, -1.06525512e-05, ...,
          2.75414138e-03,  9.26911018e-04,  0.00000000e+00]])

In [ ]:
scores = rating_matrix.dot(weight_matrix)
scores = scores - rating_matrix * 1e6
scores

matrix([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  8.93469952e-03,  1.87447796e-03, ...,
          2.14413946e-02,  4.39749619e-03,  1.15020631e-02],
        [ 0.00000000e+00,  6.21910978e-03,  9.08731273e-04, ...,
          1.31848195e-02,  1.60895389e-03,  5.95109546e-03],
        ...,
        [ 0.00000000e+00,  5.76738254e-03, -7.49698912e-05, ...,
          1.79977859e-02, -9.99999997e+05,  9.40019967e-03],
        [ 0.00000000e+00,  3.65384388e-03, -3.36212213e-05, ...,
          1.20643086e-02,  4.69897113e-03,  6.45194748e-03],
        [ 0.00000000e+00,  3.58415158e-03,  9.34465507e-04, ...,
          2.06720430e-02,  7.57318201e-03,  5.37596389e-03]])

In [ ]:
test_scores = scores[test_users]
test_scores

matrix([[ 0.00000000e+00,  8.93469952e-03,  1.87447796e-03, ...,
          2.14413946e-02,  4.39749619e-03,  1.15020631e-02],
        [ 0.00000000e+00,  6.21910978e-03,  9.08731273e-04, ...,
          1.31848195e-02,  1.60895389e-03,  5.95109546e-03],
        [ 0.00000000e+00,  1.23494388e-02,  8.62305962e-04, ...,
          2.42811663e-02,  5.13525670e-03,  1.19834874e-02],
        ...,
        [ 0.00000000e+00,  5.76738254e-03, -7.49698912e-05, ...,
          1.79977859e-02, -9.99999997e+05,  9.40019967e-03],
        [ 0.00000000e+00,  3.65384388e-03, -3.36212213e-05, ...,
          1.20643086e-02,  4.69897113e-03,  6.45194748e-03],
        [ 0.00000000e+00,  3.58415158e-03,  9.34465507e-04, ...,
          2.06720430e-02,  7.57318201e-03,  5.37596389e-03]])

In [ ]:
top_k_preds = np.argsort(-test_scores)[:, :10].tolist()
top_k_preds[-2:]

[[15, 33, 22, 14, 26, 10, 24, 35, 7, 29],
 [33, 7, 22, 15, 23, 30, 25, 24, 29, 34]]

In [ ]:
top_k_scores = -np.sort(-test_scores)[:,:10]
top_k_scores = top_k_scores.tolist()
top_k_scores[-2:]

[[0.01548683219821709,
  0.01206430857250542,
  0.01171975647780978,
  0.011413406538176255,
  0.008682750654003838,
  0.008465523338277796,
  0.007704657319093317,
  0.0064519474834630434,
  0.006435909340163184,
  0.006364914121556707],
 [0.020672043027043806,
  0.019939057772721213,
  0.019246450424837032,
  0.015008765315111191,
  0.013099674840788998,
  0.010596729435927961,
  0.00936694061037921,
  0.008285022025382652,
  0.008197090567111197,
  0.007573182012048498]]

In [ ]:
users = []
items = []
ratings = []

for i, user in enumerate(test_users):
  users.extend([user] * 10)
  items.extend(top_k_preds[i])
  ratings.extend(top_k_scores[i])

ease_preds = pd.DataFrame({'user_id': users, 'item_id': items, 'rating': ratings})

In [ ]:
ease_preds

,user_id,item_id,rating
0,1,24,0.040459
1,1,30,0.031347
2,1,5,0.025755
3,1,6,0.023698
4,1,33,0.021441
...,...,...,...
1595,160,30,0.010597
1596,160,25,0.009367
1597,160,24,0.008285
1598,160,29,0.008197


In [ ]:
ease_best_hr_K = 0
ease_best_cov_K = 0
max_hr = 0
max_cov = 0
name = 'EASE'
for k in range(1, 5):
    metrics = measure(ease_preds, test, k, name)
    if metrics.loc[name, 'hit_rate@K'] >= max_hr:
        max_hr = metrics.loc[name, 'hit_rate@K']
        ease_best_hr_K = k
    if metrics.loc[name, 'coverage@K'] >= max_cov:
        max_cov = metrics.loc[name, 'coverage@K']
        ease_best_cov_K = k

print('The best hit_rate@K:')
display(measure(ease_preds, test, ease_best_hr_K, name))

print()

print('The best coverage@K:')
display(measure(ease_preds, test, ease_best_cov_K, name))

The best hit_rate@K:


,hit_rate@K,coverage@K
EASE,0.56875,0.628571



The best coverage@K:


,hit_rate@K,coverage@K
EASE,0.56875,0.628571


### EASE Inference

In [ ]:
def recommend_from_vector(user_vector, weight_matrix, K=10):
    """
    user_vector - vector of implicit (0/1) responses
    weight_matrix - matrix of weights
    K - variable for top@k
    """

    user_vector = np.asarray(user_vector, dtype=np.float64).ravel() # приводим к 1D размеру
    scores = np.dot(user_vector, weight_matrix)
    scores = np.ravel(scores) # убеждаемся, что это 1D массив

    scores[user_vector > 0] = -np.inf # убираем уже просмотренные

    topk = np.argsort(-scores)[:K]
    return topk


In [ ]:
items_ids

defaultdict(int,
            {'Закрытие задолженностей': 1,
             'Химия': 2,
             'Наука': 3,
             'Клуб наставников': 4,
             'Глубокое обучение': 5,
             'Backend-разработка': 6,
             'Экономика': 7,
             'Frontend разработка': 8,
             'IOS-разработка': 9,
             'Музыка': 10,
             'Туризм': 11,
             'DevOps': 12,
             'Робототехника': 13,
             'Волонтерство': 14,
             'Хакатоны': 15,
             'Иностранные языки': 16,
             'Вокал': 17,
             'Android-разработка': 18,
             '3D-моделирование': 19,
             'Промышленный дизайн': 20,
             'Физика': 21,
             'Стажировки': 22,
             'Танцы': 23,
             'Машинное обучение': 24,
             'Творчество': 25,
             'Театр': 26,
             'Спортивное программирование': 27,
             'Предпринимательство': 28,
             'Интеллектуальные игры': 29,
           

In [ ]:
user_vec = np.zeros(36)
user_vec[27] = 1
user_vec[11] = 1
user_vec

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
       0., 0.])

In [ ]:
top3 = recommend_from_vector(user_vec, weight_matrix, K=3)
print("Top-3 рекомендации:", top3)

Top-3 рекомендации: [22 15 29]


In [ ]:
type(weight_matrix)

numpy.matrix

In [ ]:
np.save("weight_matrix.npy", weight_matrix)

### Test results comparison

In [ ]:
baseline_res = measure(popular_preds, test, pop_best_hr_K, 'MostPopular')
metrics = measure(ease_preds, test, ease_best_hr_K, 'EASE', baseline_res)
metrics.sort_values('hit_rate@K', ascending=False)

,hit_rate@K,coverage@K
EASE,0.56875,0.628571
MostPopular,0.4125,0.142857


Как мы видим, EASE дает лучший результат по попаданию в рекомендации нужным пользователям и в принципе является более разнообразной моделью рекомендательных систем, чем просто предложение самых популярных айтемов